In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import json
import time
from tqdm import tqdm
from curl_cffi import requests

In [2]:
URL = 'https://www.olx.ro/imobiliare/apartamente-garsoniere-de-inchiriat/bucuresti/?currency=EUR'
header = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [3]:
def get_content(URL, header):
    try:
        response = requests.get(URL,headers=header,impersonate="chrome120")
        response.raise_for_status()
        html_content = response.text
        return html_content        
    except Exception as e:
        print(f"Eroare {e}")

In [4]:
def get_listings(URL,header):
    html_content = get_content(URL,header)
    soup = BeautifulSoup(html_content,'html.parser')
    raw_listings = soup.find_all('div',attrs={'data-cy':'l-card'})
    return raw_listings

In [5]:
def get_offers (raw_listings):
    offers = []
    for listing in raw_listings:
        anchor = listing.find('a',href=True)
        if anchor:
            if anchor['href'][0:5] == 'https':
                offers.append(anchor['href'])
            else:
                url = 'https://www.olx.ro' + anchor['href']
                offers.append(url)
    return offers

In [6]:
def create_dataframe (offers):
    df = pd.DataFrame({'Link':offers})
    df['Pagina'] = df['Link'].map(lambda link: str(link).split('.')[1])
    df['Link'] = df.apply(lambda row: row['Link'] if row['Pagina'] == 'olx' else str(row['Link']).split('.html')[0], axis=1)
    return df

In [7]:
def storia_parser(URL,header):
    try:
        response = get_content(URL=URL,header=header)

        if "403 Forbidden" in response or "captcha" in response.lower():
            print("Blocked by Cloudflare/Firewall!")
        elif len(response) < 1000:
            print("Received an empty/skeleton page.")
            
        soup = BeautifulSoup(response.text,'html.parser')
        script_tag = soup.find('script', id='__NEXT_DATA__')
        json_text = json.loads(script_tag.string)
        offer_data = json_text['props']['pageProps']['ad']
        df_characteristics = pd.DataFrame(offer_data['characteristics']).set_index('key')[['localizedValue']].T
        df_characteristics['id'] = offer_data['id']
        df_characteristics['description'] = offer_data['description']
        df_characteristics['title'] = offer_data['title']
        df_characteristics['URL'] = URL
        return df_characteristics
    except Exception as e:
        print(f"{e}")

In [8]:
def olx_parser_manual(URL,header):
    try:
        html_content = get_content(URL,header)
        soup = BeautifulSoup(html_content,'html.parser')

        results = {}
        title = soup.find_all('div',attrs={'data-cy':'offer_title'})[0].find('h4').contents[0]
        results['title'] = title

        ad_price = soup.find_all('div', attrs= {'data-testid':'ad-price-container'})[0].find('h3').contents[0]
        results['ad_price'] = ad_price

        extracted_content = soup.find_all('div',attrs={'data-cy':'ad_description'})
        if extracted_content:
            container = extracted_content[0]
            for style in container.find_all("style"):
                style.decompose()
            header = container.find("h3")
            if header:
                header.decompose()
        pretty_description = container.get_text(separator="\n", strip=True)
        results['description'] = pretty_description

        listing_id = soup.find_all('div',attrs={'data-cy':'ad-footer-bar-section'})[0].find('span').contents[2]
        results['offer_id'] = listing_id
        return results
    except Exception as e:
        print(f"Exeption: {e}")

In [ ]:
raw_listings = get_listings(URL,header)
offers = get_offers(raw_listings)
df = create_dataframe(offers)

In [ ]:
results = {
    'OLX': [],
    'Storia': []
}
for idx,row in tqdm(df.iterrows()):
    if row['Pagina'] == 'olx':
        results['OLX'].append(olx_parser_manual(row['Link'],header))
    elif row['Pagina'] == 'storia':
        results['Storia'].append(storia_parser(row['Link'],header))
    time.sleep(1.5)

In [78]:
df.head()

,Link,Pagina
0,https://www.olx.ro/d/oferta/inchiriez-2-aparta...,olx
1,https://www.storia.ro/ro/oferta/apartament-2-c...,storia
2,https://www.olx.ro/d/oferta/crangasi-metrou-de...,olx
3,https://www.olx.ro/d/oferta/proprietar-inchiri...,olx
4,https://www.storia.ro/ro/oferta/garsoniera-pef...,storia


In [9]:
URL = "https://www.storia.ro/ro/oferta/apartament-2-camere-dristor-mobilat-si-utilat-complet-5-minute-met-IDGbZc"
storia_parser(URL,header)

Blocked by Cloudflare/Firewall!
'str' object has no attribute 'text'


In [15]:
def storia_parser_manual(URL, header):
    html_content = get_content(URL,header)
    soup = BeautifulSoup(html_content,'html.parser')
    
    print(html_content)
    title = soup.find_all('h1',attrs={'data-cy':'adPageAdTitle'})[0].find('h1')

    #Price section

    price_section = soup.find_all('div',attrs={'data-sentry-element':'PriceSection'})
    print(title)
    print(price_section)

In [16]:
storia_parser_manual(URL,header)

<!DOCTYPE html><html lang="ro" data-default-lang="ro" data-sentry-element="Html" data-sentry-component="CustomDocument" data-sentry-source-file="_document.tsx"><head data-sentry-element="CustomHead" data-sentry-source-file="_document.tsx"><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><link rel="icon" href="https://statics.storia.ro/static/storiaro/naspersclassifieds-regional/verticalsre-atlas-web-storiaro/static/img/favicon.ico?v=5"/><link rel="apple-touch-icon" href="https://statics.storia.ro/static/storiaro/naspersclassifieds-regional/verticalsre-atlas-web-storiaro/static/img/app-icon.png"/><link rel="android-touch-icon" href="https://statics.storia.ro/static/storiaro/naspersclassifieds-regional/verticalsre-atlas-web-storiaro/static/img/app-icon.png"/><meta property="fb:app_id" content="1758851897729052"/><meta property="og:type" content="website"/><meta property="og:site_name" content="www.storia.ro/"/><meta property="al:web:url" content="https://www.sto